# 🎧 MagL+X: PyTorch Loss Sonification (Runnable Notebook)

This notebook demonstrates how to:
- Train a simple PyTorch model
- Extract per-sample losses
- Convert them into audio
- Stream real-time sound using MagL+X concepts

---


In [ ]:
# The notebook lives in examples/ but the modules (synth.py, live_engine.py)
# are at the repo root, so put the root on sys.path before importing them.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch numpy sounddevice

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import sounddevice as sd

## 🔊 Synth Engine

In [ ]:
from synth import Synth

synth = Synth()

## 🎛 Live Audio Engine

In [ ]:
from live_engine import (
    LiveEngine,
    CrossEntropyTracker,   # mismatch between consecutive loss distributions
    WavetableTracker,      # NEW: loss magnitude -> volume + timbre
)
from synth import SR, BLOCK

# LiveEngine takes the sample rate and block size explicitly.
engine = LiveEngine(SR, BLOCK)


## 🧠 Loss → Audio Mapping

In [ ]:
# Two independent sonifications are available. Pick either (or both).
#
#   CrossEntropyTracker -> hears the *mismatch* between consecutive
#                          loss distributions (how much the model CHANGED).
#   WavetableTracker    -> hears the loss *magnitude* (how WRONG it is):
#                          low loss  = quiet, pure sine
#                          high loss = loud, sawtooth

# mode="classic" is the original sound; mode="wavetable" morphs P's timbre
# toward Q's by the mismatch itself. Classic is the default.
tracker = CrossEntropyTracker(mode="classic")

wave_tracker = WavetableTracker()   # self-calibrating against a running max


## 🧪 Toy Dataset

In [ ]:
# Simple synthetic classification data
X = torch.randn(200, 10)
y = (X.sum(dim=1) > 0).long()

## 🤖 Model

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(reduction='none')

## ▶️ Training with Live Audio

In [ ]:
engine.start()

# Choose what you want to hear by setting SONIFICATION:
#   "loss"          -> WavetableTracker: volume + sine->saw timbre (Path A)
#   "xent"          -> CrossEntropyTracker, classic sound
#   "xent_wavetable"-> CrossEntropyTracker, P->Q timbre morph
SONIFICATION = "loss"

for epoch in range(5):
    logits = model(X)
    loss = criterion(logits, y)

    optimizer.zero_grad()
    loss.mean().backward()
    optimizer.step()

    # NOTE: every hook takes the synth explicitly.
    if SONIFICATION == "loss":
        audio_fn = wave_tracker.to_audio_fn(loss, synth)
    elif SONIFICATION == "xent_wavetable":
        audio_fn = tracker.to_audio_fn(loss, synth, mode="wavetable")
    else:
        audio_fn = tracker.to_audio_fn(loss, synth)

    engine.current_audio_fn = audio_fn

    print(f"Epoch {epoch}, loss = {loss.mean().item():.4f}")

engine.stop()


---
## 🌊 Wavetable Synthesis

Two new sounds, both optional — the original sounds are unchanged and
remain the default, so you can A/B them.

**1. Loss → volume + timbre (`loss_to_sound`)**

| loss | volume | waveform |
|------|--------|----------|
| low  | quiet  | pure sine |
| high | loud   | sawtooth |

Rather than crossfading two oscillators, the table *morphs*: harmonic 1
stays fixed and harmonics 2..N fade in at amplitude `w/n`, so `w=0` is
exactly a sine and `w=1` is a band-limited sawtooth. Every value between
is a real timbral position.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# One second of each morph position, so you can hear sine -> saw.
for w in [0.0, 0.5, 1.0]:
    sig = synth.wavetable(220.0, 0.8, w, SR, voice=f"demo{w}")
    print(f"w = {w}  ({'pure sine' if w==0 else 'sawtooth' if w==1 else 'halfway'})")
    display(Audio(sig, rate=SR))


In [ ]:
# The wavetables themselves: one cycle at each morph position.
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for a, w in zip(ax, [0.0, 0.5, 1.0]):
    a.plot(synth._morph_table(w, 220.0), lw=1)
    a.set_title(f"w = {w}")
    a.set_xticks([]); a.set_yticks([])
plt.suptitle("sine -> sawtooth morph")
plt.tight_layout(); plt.show()


In [ ]:
# The mapping end to end: loss drives BOTH volume and timbre.
for loss_val in [0.05, 0.5, 2.0]:
    sig = synth.loss_to_sound(loss_val, SR, loss_scale=2.0,
                              voice=f"L{loss_val}")
    rms = float(np.sqrt(np.mean(sig ** 2)))
    print(f"loss = {loss_val:4.2f}   rms = {rms:.3f}")
    display(Audio(sig, rate=SR))


### 🎭 Wavetable `xent`: hearing one structure pulled toward another

The classic `xent` sound reads the mismatch as a *number* and dials
modulation and noise with it. The wavetable version makes the mismatch
**structural** instead:

- `P` and `Q` are each built into a wavetable — their weights *are* the
  harmonic amplitudes (MagL's "distribution as spectrum" idea).
- The normalized Jensen–Shannon divergence `d` sets how far `P`'s table is
  morphed toward `Q`'s: `table = (1-d)·table_P + d·table_Q`.

At `d = 0` the morph **does not move at all** — you hear `P`, untouched.
Silence-at-convergence becomes structural rather than a scalar turned down.
As `d` rises you hear one structure literally deformed into the other,
which is what `emit xent P Q` means.


In [ ]:
P = [0.2, 0.3, 0.5]
Q = [0.4, 0.4, 0.2]

d = synth.js_divergence(P, Q) / np.log(2)
print(f"normalized mismatch d = {d:.4f}\n")

print("classic  (spectrum of P + modulation/noise scaled by d):")
display(Audio(synth.xent_sound(P, Q, SR, mode="classic"), rate=SR))

print("wavetable (P's timbre morphed toward Q's by d):")
display(Audio(synth.xent_sound(P, Q, SR, mode="wavetable"), rate=SR))

print("identical distributions -- the morph does not move:")
display(Audio(synth.xent_sound(P, P, SR, mode="wavetable"), rate=SR))


In [ ]:
# See the morph: P's table, the blend at d, and Q's table.
tP = synth._table_from_distribution(P, 220.0)
tQ = synth._table_from_distribution(Q, 220.0)

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
ax[0].plot(tP, lw=1);            ax[0].set_title("table P")
ax[1].plot((1-d)*tP + d*tQ, lw=1); ax[1].set_title(f"morphed at d = {d:.3f}")
ax[2].plot(tQ, lw=1);            ax[2].set_title("table Q")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.suptitle("the mismatch IS the distance the timbre travels")
plt.tight_layout(); plt.show()
